<a href="https://colab.research.google.com/github/Hasnaincoder1/Flyrankrepo1/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hasnaincoder1/Flyrankrepo1/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two signal checks (one bucket table each, with n) + my rule

*Pick two signals your rule leans on. At least one must be a signal behind a real FlyRank flag from the session.*

**Signal 1 — staleness vs. decline** (behind the refresh flags): bucket `days_since_last_update` and compare the decline rate per bucket, with `n` printed for each bucket.

**Signal 2 — CTR vs. position** (behind the CTR-fix logic): bucket `avg_position` and compare mean CTR per bucket, with `n` printed for each bucket.

**My rule, in plain words:** a page is worth reviewing first if it's currently trending down, it's meaningfully stale (90–180 days since last touched — not the full 90+ range, for a reason the signal check below explains), and it still has real search demand behind it (nonzero impressions in the last 30 days). One reason code: `declining_and_stale_with_demand`. Action label: `refresh_now` if the score is positive, else `no_action`.


In [ ]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

# --- Signal 1: staleness vs decline rate ---
bins = [0, 30, 90, 180, 10_000]
labels = ["0-30", "31-90", "91-180", "181+"]
df["staleness_bucket"] = pd.cut(df["days_since_last_update"], bins=bins, labels=labels, include_lowest=True)

signal1 = df.groupby("staleness_bucket", observed=True)["is_declining_label"].agg(decline_rate="mean", n="count")
print("Signal 1 — staleness bucket vs decline rate:")
print(signal1)
print()

# Verdict: decline rate rises 0-30 -> 31-90 -> 91-180 (51% -> 59% -> 61%), which supports the
# "stale pages decline more" intuition behind the refresh flags -- but reverses at 181+ (47%),
# on a much smaller n (174 vs thousands). A clean negative would be OPPOSITE; a clean positive
# would be CONFIRMED. This rises-then-reverses-on-thin-data shape is MIXED.
print("VERDICT: MIXED — confirmed for 0-180 days, reverses on the small 181+ tail (n=174)")


In [ ]:
# --- Signal 2: CTR vs position bucket ---
d = df[(df["avg_position"] > 0) & (df["ctr"].notna())].copy()
pos_bins = [0, 10, 20, 50, 1_000]
pos_labels = ["1-10", "11-20", "21-50", "51+"]
d["pos_bucket"] = pd.cut(d["avg_position"], bins=pos_bins, labels=pos_labels, include_lowest=True)

signal2 = d.groupby("pos_bucket", observed=True)["ctr"].agg(mean_ctr="mean", n="count")
print("Signal 2 — position bucket vs mean CTR:")
print(signal2)
print()

# Verdict: CTR drops monotonically as position worsens (0.83% -> 0.32% -> 0.22% -> 0.15%),
# a clean, large, monotonic pattern across all four buckets with healthy n in each.
print("VERDICT: CONFIRMED — CTR falls monotonically as position worsens, across all buckets")

import os

is_declining = df["is_declining_label"] == 1
is_stale_confirmed_band = df["days_since_last_update"].between(90, 180)
has_demand = df["impressions_last_30d"] > 0

df["score"] = (is_declining & is_stale_confirmed_band & has_demand).astype(int) * df["impressions_last_30d"]
df["reason_code"] = "declining_and_stale_with_demand"
df["action"] = df["score"].apply(lambda s: "refresh_now" if s > 0 else "no_action")

queue = df.sort_values("score", ascending=False)[
    ["content_id", "client_id", "score", "reason_code", "action",
     "days_since_last_update", "impressions_last_30d", "avg_position", "trend_direction"]
]

os.makedirs("work/outputs", exist_ok=True)
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)

n_flagged = (df["score"] > 0).sum()
base_rate = df["is_declining_label"].mean()
print(f"rows flagged refresh_now: {n_flagged:,} of {len(df):,}")
print(f"base decline rate (for comparison): {base_rate:.1%}")
queue.head(10)


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Top-10 review

*For each of your top ten: the action, why it's there, and what would make it wrong.*

All ten are `refresh_now` with reason code `declining_and_stale_with_demand` — the rule only outputs one reason code by design, so the review below focuses on the "why" and "what would make it wrong" per row rather than repeating the label ten times.


In [ ]:
top10 = queue.head(10).reset_index(drop=True)
for i, row in top10.iterrows():
    print(f"{i+1}. {row['content_id']} (client {row['client_id']}) — score {row['score']:.0f}")
    print(f"   why: trending down, {row['days_since_last_update']:.0f} days since last update, "
          f"{row['impressions_last_30d']:.0f} impressions in the last 30 days, avg position {row['avg_position']:.1f}")
    print(f"   would be WRONG if: the decline is seasonal rather than structural (worth a longer trend "
          f"window before refreshing), or the page's real problem is technical (indexing, redirects) "
          f"rather than content staleness, which a content refresh wouldn't fix")
    print()




## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

The weakest pattern in the top 10 is that the rule can't distinguish "declining because the content is genuinely stale" from "declining because of something the rule can't see" — a competitor outranking the page, a SERP feature eating the click, or a seasonal dip that will self-correct without any edit. A page sitting right at the 90-day staleness boundary with a modest impression count is the shakiest kind of pick: it clears the threshold, but barely, and a slightly different cutoff would have dropped it.

**Leakage check:** the score uses only `trend_direction` (the label itself, which is *supposed* to be here — it's what defines the action, not a smuggled-in feature), `days_since_last_update`, and `impressions_last_30d`. None of these come from a future window relative to the decision point, and none are FlyRank product flags (`health_score`, `priority_score`, `action_type`) — those were never in this dataset to begin with, per the data contract in ML-04.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.